In [8]:
import duckdb
import numpy as np
from pathlib import Path

In [ ]:
# import duckdb
# import multiprocessing

# # -------------------------------
# # SETTINGS
# # -------------------------------

# TRANSACTIONS_PATH = "transactions/**/*.parquet"
# ADDITIONAL_PATH = "transactions_additional/**/*.parquet"

# OUTPUT_DB = "aml_features.duckdb"
# OUTPUT_PARQUET = "transaction_feature4.parquet"

# THREADS = max(1, multiprocessing.cpu_count()-1)

# # -------------------------------
# # CONNECT DUCKDB
# # -------------------------------

# con = duckdb.connect(OUTPUT_DB)

# con.execute(f"PRAGMA threads={THREADS}")
# con.execute("PRAGMA memory_limit='32GB'")
# con.execute("PRAGMA temp_directory='duck_temp'")
# con.execute("PRAGMA enable_progress_bar")

# # -------------------------------
# # FEATURE ENGINEERING QUERY
# # -------------------------------

# sql = """

# WITH transactions AS (
# SELECT * FROM read_parquet('transactions/**/*.parquet')
# ),

# additional AS (
# SELECT * FROM read_parquet('transactions_additional/**/*.parquet')
# ),

# base AS (

# SELECT
# t.transaction_id,
# t.account_id,
# CAST(t.transaction_timestamp AS TIMESTAMP) AS ts,
# t.amount,
# t.txn_type,
# t.channel,

# a.latitude,
# a.longitude,
# a.ip_address,
# a.balance_after_transaction,
# a.part_transaction_type,
# a.atm_deposit_channel_code,
# a.transaction_sub_type

# FROM transactions t
# LEFT JOIN additional a
# USING(transaction_id)

# ),

# -- GEO FEATURES

# geo AS (

# SELECT
# account_id,

# COUNT(*) FILTER (WHERE latitude IS NOT NULL) AS geo_txn_count,

# COUNT(*) FILTER (WHERE latitude IS NOT NULL) * 1.0 / COUNT(*) AS geo_frac,

# COUNT(DISTINCT latitude || longitude) AS unique_geo_count,

# CASE WHEN COUNT(*) FILTER (WHERE latitude IS NOT NULL) > 0
# THEN 1 ELSE 0 END AS has_geo_flag,

# MIN(latitude) AS min_lat,
# MAX(latitude) AS max_lat,
# MIN(longitude) AS min_lon,
# MAX(longitude) AS max_lon

# FROM base
# GROUP BY account_id
# ),

# geo_bbox AS (

# SELECT
# account_id,

# 6371 * 2 * ASIN(
# SQRT(
# POWER(SIN(RADIANS((max_lat-min_lat)/2)),2) +
# COS(RADIANS(min_lat))*COS(RADIANS(max_lat)) *
# POWER(SIN(RADIANS((max_lon-min_lon)/2)),2)
# )
# ) AS geo_bbox_km

# FROM geo
# ),

# -- IP FEATURES

# ip_counts AS (

# SELECT
# account_id,
# ip_address,
# COUNT(*) AS ip_count

# FROM base
# WHERE ip_address IS NOT NULL
# GROUP BY account_id, ip_address

# ),

# ip_top AS (

# SELECT
# account_id,
# MAX(ip_count) AS top_ip_count

# FROM ip_counts
# GROUP BY account_id

# ),

# ip_basic AS (

# SELECT

# account_id,

# COUNT(*) FILTER (WHERE ip_address IS NOT NULL) AS ip_txn_count,

# COUNT(*) FILTER (WHERE ip_address IS NOT NULL)*1.0 / COUNT(*) AS ip_frac,

# COUNT(DISTINCT ip_address) AS unique_ip_count

# FROM base
# GROUP BY account_id

# ),

# -- ATM FEATURES

# atm AS (

# SELECT

# account_id,

# COUNT(*) FILTER (WHERE atm_deposit_channel_code IN ('CDM','CRM')) AS atm_deposit_count,

# COUNT(*) FILTER (WHERE atm_deposit_channel_code IN ('CDM','CRM'))*1.0/COUNT(*) AS atm_deposit_frac,

# STRING_AGG(DISTINCT atm_deposit_channel_code, ',') AS atm_deposit_types

# FROM base
# GROUP BY account_id

# ),

# -- BALANCE FEATURES

# balance_stats AS (

# SELECT

# account_id,

# AVG(balance_after_transaction) AS avg_balance,
# MIN(balance_after_transaction) AS min_balance,
# MAX(balance_after_transaction) AS max_balance,

# AVG(balance_after_transaction) FILTER (WHERE txn_type='C') AS avg_balance_credit,
# AVG(balance_after_transaction) FILTER (WHERE txn_type='D') AS avg_balance_debit

# FROM base
# GROUP BY account_id

# ),

# -- SUBTYPE RATIOS

# ratios AS (

# SELECT

# account_id,

# AVG(CASE WHEN transaction_sub_type='CLT_CASH' THEN 1 ELSE 0 END) AS clt_cash_ratio,
# AVG(CASE WHEN transaction_sub_type='LOAN' THEN 1 ELSE 0 END) AS loan_ratio,
# AVG(CASE WHEN transaction_sub_type='NORMAL' THEN 1 ELSE 0 END) AS normal_ratio,

# AVG(CASE WHEN part_transaction_type='CI' THEN 1 ELSE 0 END) AS ci_ratio,
# AVG(CASE WHEN part_transaction_type='BI' THEN 1 ELSE 0 END) AS bi_ratio,
# AVG(CASE WHEN part_transaction_type='IP' THEN 1 ELSE 0 END) AS ip_ratio,
# AVG(CASE WHEN part_transaction_type='IC' THEN 1 ELSE 0 END) AS ic_ratio

# FROM base
# GROUP BY account_id

# ),

# -- IP HOURLY FEATURES

# ip_hour AS (

# SELECT
# account_id,
# date_trunc('hour', ts) AS hr,
# COUNT(DISTINCT ip_address) AS unique_ips_hour

# FROM base
# WHERE ip_address IS NOT NULL
# GROUP BY account_id, hr

# ),

# ip_hour_stats AS (

# SELECT

# account_id,

# AVG(unique_ips_hour) AS avg_unique_ip_hour,
# MAX(unique_ips_hour) AS max_unique_ip_hour

# FROM ip_hour
# GROUP BY account_id

# ),

# -- IP DAILY FEATURES

# ip_day AS (

# SELECT
# account_id,
# date_trunc('day', ts) AS dy,
# COUNT(DISTINCT ip_address) AS unique_ips_day

# FROM base
# WHERE ip_address IS NOT NULL
# GROUP BY account_id, dy

# ),

# ip_day_stats AS (

# SELECT

# account_id,

# AVG(unique_ips_day) AS avg_unique_ip_day,
# MAX(unique_ips_day) AS max_unique_ip_day

# FROM ip_day
# GROUP BY account_id

# )

# SELECT

# g.account_id,

# g.geo_txn_count,
# g.geo_frac,
# g.unique_geo_count,
# gb.geo_bbox_km,
# g.has_geo_flag,

# ip.ip_txn_count,
# ip.ip_frac,
# ip.unique_ip_count,

# t.top_ip_count,
# t.top_ip_count*1.0/ip.ip_txn_count AS top_ip_share,

# atm.atm_deposit_count,
# atm.atm_deposit_frac,
# atm.atm_deposit_types,

# b.avg_balance,
# b.min_balance,
# b.avg_balance_credit,
# b.avg_balance_debit,
# b.max_balance,

# r.clt_cash_ratio,
# r.loan_ratio,
# r.normal_ratio,
# r.ci_ratio,
# r.bi_ratio,
# r.ip_ratio,
# r.ic_ratio,

# h.avg_unique_ip_hour,
# h.max_unique_ip_hour,

# d.avg_unique_ip_day,
# d.max_unique_ip_day

# FROM geo g

# LEFT JOIN geo_bbox gb USING(account_id)
# LEFT JOIN ip_basic ip USING(account_id)
# LEFT JOIN ip_top t USING(account_id)
# LEFT JOIN atm USING(account_id)
# LEFT JOIN balance_stats b USING(account_id)
# LEFT JOIN ratios r USING(account_id)
# LEFT JOIN ip_hour_stats h USING(account_id)
# LEFT JOIN ip_day_stats d USING(account_id)

# """

# # -------------------------------
# # CREATE FEATURE TABLE
# # -------------------------------

# con.execute("DROP TABLE IF EXISTS transaction_feature4")

# con.execute("CREATE TABLE transaction_feature4 AS " + sql)

# # -------------------------------
# # EXPORT PARQUET
# # -------------------------------

# con.execute(f"""
# COPY transaction_feature4
# TO '{OUTPUT_PARQUET}'
# (FORMAT PARQUET)
# """)

# print("✅ Feature table created")
# print("📁 Saved as:", OUTPUT_PARQUET)

# con.close()

✅ Feature table created
📁 Saved as: transaction_feature4.parquet


In [15]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [19]:
import duckdb

con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('transactions_additional/**/*.parquet')
USING SAMPLE 10 ROWS
""").df()

df

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0000003009,UPD,NaN,NaN,122.41.216.127,-9314953.86,CI,None,normal
1,TXN_0000004799,UPD,NaN,NaN,122.210.14.206,-31318190.49,CI,None,normal
2,TXN_0000013229,UPC,NaN,NaN,None,13469850.24,CI,None,normal
3,TXN_0000016444,UPC,30.280983,78.009819,None,-23179456.14,CI,None,normal
4,TXN_0000019430,CHQ,NaN,NaN,None,-7396622.82,CI,None,normal
5,TXN_0000023756,MAD,NaN,NaN,None,-2186564.81,CI,None,normal
6,TXN_0000025134,UPC,25.447967,81.833220,None,2220293.33,CI,None,normal
7,TXN_0000034699,UPD,NaN,NaN,122.135.186.236,-5084003.34,CI,None,normal
8,TXN_0000036956,TPC,NaN,NaN,None,-15762867.80,CI,None,normal
9,TXN_0000039260,UPD,31.343197,75.573258,None,-6158891.19,CI,None,normal


In [17]:
import pandas as pd
df = pd.read_parquet("transaction_feature4.parquet")
df.sample(10)

,account_id,geo_txn_count,geo_frac,unique_geo_count,geo_bbox_km,has_geo_flag,ip_txn_count,ip_frac,unique_ip_count,top_ip_count,top_ip_share,atm_deposit_count,atm_deposit_frac,atm_deposit_types,avg_balance,min_balance,avg_balance_credit,avg_balance_debit,max_balance,clt_cash_ratio,loan_ratio,normal_ratio,ci_ratio,bi_ratio,ip_ratio,ic_ratio,avg_unique_ip_hour,max_unique_ip_hour,avg_unique_ip_day,max_unique_ip_day
148482,ACCT_030357,434,0.377063,434,15.227500,1,440,0.382276,440,1.0,0.002273,2,0.001738,CDM,-9.701985e+06,-14739832.87,-9.767645e+06,-9.642521e+06,49178.18,0.0,0.0,0.0,0.950478,0.025195,0.024327,0.0,1.016166,2.0,1.268012,4.0
70025,ACCT_009152,1163,0.288514,1163,14.911793,1,1460,0.362193,1460,1.0,0.000685,3,0.000744,"CDM,CRM",2.803058e+06,-4999991.16,2.778152e+06,2.824037e+06,11421349.51,0.0,0.0,0.0,0.963285,0.030265,0.006450,0.0,1.050360,3.0,2.249615,7.0
109488,ACCT_153048,419,0.325311,419,15.296820,1,426,0.330745,426,1.0,0.002347,2,0.001553,CDM,1.951762e+06,-1107495.60,1.941238e+06,1.960826e+06,4949056.81,0.0,0.0,0.0,0.944099,0.034161,0.021739,0.0,1.016706,2.0,1.310769,4.0
23140,ACCT_145366,692,0.433041,692,14.730690,1,565,0.353567,565,1.0,0.001770,7,0.004380,"CRM,CDM",-5.662095e+06,-13524897.95,-5.635859e+06,-5.686188e+06,4109848.12,0.0,0.0,0.0,0.947434,0.033166,0.019399,0.0,1.014363,2.0,1.292906,4.0
140962,ACCT_128868,360,0.324324,360,14.480705,1,361,0.325225,361,1.0,0.002770,4,0.003604,"CRM,CDM",-1.246307e+05,-1857937.31,-1.324288e+05,-1.168045e+05,1804328.59,0.0,0.0,0.0,0.929730,0.030631,0.039640,0.0,1.002778,2.0,1.164516,4.0
66737,ACCT_079068,769,0.380317,769,14.763758,1,699,0.345697,699,1.0,0.001431,6,0.002967,"CRM,CDM",-4.864477e+06,-8357482.93,-4.775974e+06,-4.940150e+06,1770182.05,0.0,0.0,0.0,0.966864,0.032146,0.000989,0.0,1.373281,5.0,11.274194,25.0
65421,ACCT_017450,1475,0.376468,1475,14.717695,1,1412,0.360388,1412,1.0,0.000708,7,0.001787,"CDM,CRM",2.005548e+07,-1210554.77,1.977299e+07,2.030130e+07,48959225.76,0.0,0.0,0.0,0.965288,0.029096,0.005615,0.0,1.068079,3.0,2.494700,9.0
63420,ACCT_003672,360,0.342205,360,14.496671,1,356,0.338403,356,1.0,0.002809,1,0.000951,CRM,2.841424e+06,29203.01,2.829528e+06,2.852324e+06,3920068.39,0.0,0.0,0.0,0.920152,0.029468,0.050380,0.0,1.005650,2.0,1.126582,3.0
23010,ACCT_075061,2092,0.368310,2092,1427.469524,1,1882,0.331338,1882,1.0,0.000531,17,0.002993,"CRM,CDM",-1.285563e+07,-23870540.09,-1.279899e+07,-1.290725e+07,2146839.09,0.0,0.0,0.0,0.955986,0.039789,0.004225,0.0,1.077275,3.0,2.804769,10.0
130460,ACCT_064537,630,0.385793,630,14.765129,1,542,0.331904,542,1.0,0.001845,4,0.002449,"CRM,CDM",1.047593e+06,-3614096.19,9.137165e+05,1.163283e+06,31429839.79,0.0,0.0,0.0,0.951623,0.039192,0.009186,0.0,1.028463,2.0,1.672840,6.0


In [24]:
import duckdb

con = duckdb.connect()

con.execute("""
COPY (
    SELECT
        * REPLACE (
            CAST(ROUND(avg_unique_ip_day) AS INTEGER) AS avg_unique_ip_day,
            CAST(ROUND(avg_unique_ip_hour) AS INTEGER) AS avg_unique_ip_hour
        )
    FROM 'transaction_feature4.parquet'
)
TO 'transaction_features_4.parquet'
(FORMAT PARQUET);
""")

In [25]:
import pandas as pd
df = pd.read_parquet("transaction_features_4.parquet")
df.sample(10)

,account_id,geo_txn_count,geo_frac,unique_geo_count,geo_bbox_km,has_geo_flag,ip_txn_count,ip_frac,unique_ip_count,top_ip_count,top_ip_share,atm_deposit_count,atm_deposit_frac,atm_deposit_types,avg_balance,min_balance,avg_balance_credit,avg_balance_debit,max_balance,clt_cash_ratio,loan_ratio,normal_ratio,ci_ratio,bi_ratio,ip_ratio,ic_ratio,avg_unique_ip_hour,max_unique_ip_hour,avg_unique_ip_day,max_unique_ip_day
27521,ACCT_089438,1716,0.363328,1716,15.186634,1,1648,0.348931,1648,1.0,0.000607,13,0.002752,"CRM,CDM",-1.660474e+07,-26718793.45,-1.650630e+07,-1.669460e+07,4882609.90,0.0,0.0,0.0,0.957019,0.033665,0.009316,0.000000,1.0,3.0,2.0,6.0
80274,ACCT_149824,264,0.344648,264,1540.444951,1,233,0.304178,233,1.0,0.004292,1,0.001305,CDM,-1.545100e+06,-3873229.59,-1.436555e+06,-1.627255e+06,64507.36,0.0,0.0,0.0,0.929504,0.032637,0.000000,0.037859,1.0,2.0,1.0,3.0
65405,ACCT_151181,621,0.348681,621,15.373457,1,612,0.343627,612,1.0,0.001634,12,0.006738,"CDM,CRM",1.484902e+06,-2017064.28,1.522098e+06,1.451473e+06,4096563.75,0.0,0.0,0.0,0.956204,0.032004,0.011791,0.000000,1.0,3.0,2.0,5.0
132670,ACCT_069274,363,0.261339,363,1448.373946,1,441,0.317495,441,1.0,0.002268,3,0.002160,CDM,3.551534e+07,-3059782.90,3.556499e+07,3.546648e+07,53248117.13,0.0,0.0,0.0,0.940245,0.038157,0.021598,0.000000,1.0,2.0,1.0,5.0
100451,ACCT_078892,781,0.371728,781,792.181410,1,735,0.349833,735,1.0,0.001361,6,0.002856,"CRM,CDM",-8.338777e+06,-13434486.19,-8.376120e+06,-8.307478e+06,24961.02,0.0,0.0,0.0,0.949072,0.038077,0.012851,0.000000,1.0,2.0,2.0,5.0
54413,ACCT_152473,714,0.355401,714,445.506427,1,697,0.346939,697,1.0,0.001435,5,0.002489,CDM,-1.422364e+06,-4511708.55,-1.388980e+06,-1.452431e+06,858006.30,0.0,0.0,0.0,0.943255,0.026879,0.029866,0.000000,1.0,2.0,1.0,4.0
72440,ACCT_079970,261,0.343874,261,282.889810,1,218,0.287220,218,1.0,0.004587,2,0.002635,CDM,-1.424090e+07,-18326749.62,-1.444925e+07,-1.404273e+07,230890.16,0.0,0.0,0.0,0.885375,0.035573,0.079051,0.000000,1.0,1.0,1.0,4.0
96846,ACCT_048804,961,0.258403,961,15.233125,1,1272,0.342027,1272,1.0,0.000786,6,0.001613,"CDM,CRM",-5.976614e+06,-42108641.40,-6.164471e+06,-5.807841e+06,4136453.14,0.0,0.0,0.0,0.948642,0.035225,0.016133,0.000000,1.0,2.0,1.0,6.0
97805,ACCT_002032,500,0.260417,500,15.214201,1,669,0.348438,669,1.0,0.001495,5,0.002604,"CDM,CRM",2.883517e+06,-1794697.33,2.748220e+06,2.998964e+06,11066654.33,0.0,0.0,0.0,0.963021,0.032813,0.004167,0.000000,1.0,3.0,3.0,8.0
23057,ACCT_008267,276,0.289308,276,15.121642,1,316,0.331237,316,1.0,0.003165,4,0.004193,"CDM,CRM",-1.080497e+07,-13778739.58,-1.065709e+07,-1.093156e+07,4019.99,0.0,0.0,0.0,0.949686,0.042977,0.007338,0.000000,1.0,3.0,2.0,5.0


In [23]:
df.shape

(159776, 30)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159776 entries, 0 to 159775
Data columns (total 30 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   account_id          159776 non-null  object 
 1   geo_txn_count       159776 non-null  int64  
 2   geo_frac            159776 non-null  float64
 3   unique_geo_count    159776 non-null  int64  
 4   geo_bbox_km         156614 non-null  float64
 5   has_geo_flag        159776 non-null  int32  
 6   ip_txn_count        159776 non-null  int64  
 7   ip_frac             159776 non-null  float64
 8   unique_ip_count     159776 non-null  int64  
 9   top_ip_count        159385 non-null  float64
 10  top_ip_share        159385 non-null  float64
 11  atm_deposit_count   159776 non-null  int64  
 12  atm_deposit_frac    159776 non-null  float64
 13  atm_deposit_types   148856 non-null  object 
 14  avg_balance         159776 non-null  float64
 15  min_balance         159776 non-nul

merging 

In [28]:
df1 = pd.read_parquet("transaction_features.parquet")
print(df.shape)
df1.sample(6)

(159776, 30)


,account_id,total_transactions,unique_mcc_count,most_common_mcc,max_mcc_frequency,mcc_max_amount,mcc_min_amount,unique_channel_count,most_common_channel,max_channel_frequency,channel_max_amount,channel_min_amount,avg_unique_mcc_per_hour,avg_unique_mcc_per_day,avg_unique_channel_per_hour,avg_unique_channel_per_day,least_common_channel,least_common_mcc_code
157137,ACCT_114579,1492,54,9384,299,5905,4106,33,UPC,576,UPD,UPC,1,1,1,2,ASD,8999
30226,ACCT_110403,2492,53,9384,506,2017,5905,35,UPC,939,UPD,UPC,1,1,1,2,CTC,4900
84776,ACCT_019900,1969,54,9384,415,2017,2955,34,UPC,741,UPD,UPC,1,2,1,2,ASD,6051
23629,ACCT_132203,1678,53,9384,355,4702,9355,34,UPC,611,UPD,UPC,1,2,1,2,CTC,5311
66129,ACCT_117520,1274,48,9384,290,2557,5905,33,UPC,479,UPD,UPD,1,2,1,2,CCL,5933
75222,ACCT_108641,1371,47,9384,326,5905,5651,32,UPD,504,IPM,UPD,1,1,1,2,ASD,6211


In [29]:
df2 = pd.read_parquet("transaction_features_2.parquet")
print(df2.shape)
df2.sample(6)

(159776, 27)


,account_id,total_transaction_amount,avg_transaction_amount,max_transaction_amount,min_transaction_amount,std_transaction_amount,median_transaction_amount,total_credit_amount,total_debit_amount,max_credit_amount,max_debit_amount,credit_debit_ratio,credit_ratio,debit_ratio,avg_time_between_txn_sec,median_time_between_txn_sec,max_txn_1hr,max_txn_10min,max_txn_1day,max_txn_1week,avg_txn_per_day,avg_txn_per_hr,avg_txn_per_week,avg_txn_per_month,night_txn_rate,day_txn_rate,rapid_passthrough_count
40286,ACCT_002980,59270127.96,30662.249333,10347094.04,-150500.00,322455.095731,891.38,40344535.79,18925592.17,10347094.04,1630638.75,2.131745,0.458355,0.541645,25311.778468,13260.0,5,3,10,35,3.409171,0.142289,23.864198,101.736842,0.052768,0.933264,11.0
129512,ACCT_043329,39301523.86,29307.624057,4211000.00,-34000.00,199897.246747,1000.00,19266959.79,20034564.07,2536100.36,4211000.00,0.961686,0.483967,0.516033,54697.723134,37356.0,3,2,6,20,1.579505,0.065861,10.991803,47.892857,0.052946,0.929157,1.0
24125,ACCT_184420,25970881.31,30233.854843,5302000.00,-6000.00,231405.962008,900.00,15534380.36,10436500.95,5302000.00,2867500.00,1.488466,0.473807,0.526193,77386.479021,59013.0,2,2,7,16,1.117035,0.046571,7.809091,33.038462,0.072177,0.909197,NaN
129986,ACCT_087724,6353140.22,13721.685140,835000.00,-942.26,64946.581099,450.16,3423510.02,2929630.20,835000.00,490000.00,1.168581,0.522678,0.477322,339922.231602,251391.5,2,1,3,5,0.254535,0.010613,1.780769,7.716667,0.038877,0.946004,1.0
85897,ACCT_049613,25353322.86,22416.731088,2809200.00,-3785.19,135454.725648,1000.00,9799743.02,15553579.84,1824500.00,2809200.00,0.630064,0.456233,0.543767,139396.285841,94180.0,2,2,6,15,0.620066,0.025848,4.333333,18.850000,0.048630,0.932803,NaN
36953,ACCT_136999,9823182.77,22376.270547,2071156.45,-164166.34,125388.657057,1000.00,5587274.72,4235908.05,2071156.45,569367.68,1.319026,0.496583,0.503417,146487.561644,94007.0,2,2,4,10,0.590848,0.024630,4.102804,17.560000,0.050114,0.924829,1.0


In [30]:
df3 = pd.read_parquet("transaction_features_3.parquet")
print(df3.shape)
df3.sample(6)

(159776, 11)


,account_id,unique_counterparty_count,repeated_counterparty_frequency,unique_credit_counterparty_count,unique_debit_counterparty_count,avg_unique_counterparty_1hr,max_unique_counterparty_1hr,avg_repeated_counterparty_freq_1hr,avg_unique_counterparty_1day,max_unique_counterparty_1day,avg_repeated_counterparty_freq_1day
126132,ACCT_170309,51,0.924668,36,19,1.008942,2,0.000000,1.200371,5,0.024583
144698,ACCT_053470,13,0.988455,13,12,1.037893,3,0.001386,1.743044,6,0.034343
56065,ACCT_157346,23,0.994665,22,23,1.246856,4,0.005874,6.738050,14,0.155489
156056,ACCT_188107,29,0.982054,29,28,1.055665,3,0.001310,2.159501,7,0.022868
25864,ACCT_141343,27,0.992781,27,26,1.153369,4,0.003544,4.454902,13,0.068900
14938,ACCT_088850,19,0.988575,19,18,1.060858,4,0.002135,2.188889,7,0.034635


In [31]:
df4 = pd.read_parquet("transaction_features_4.parquet")
print(df4.shape)
df4.sample(6)

(159776, 30)


,account_id,geo_txn_count,geo_frac,unique_geo_count,geo_bbox_km,has_geo_flag,ip_txn_count,ip_frac,unique_ip_count,top_ip_count,top_ip_share,atm_deposit_count,atm_deposit_frac,atm_deposit_types,avg_balance,min_balance,avg_balance_credit,avg_balance_debit,max_balance,clt_cash_ratio,loan_ratio,normal_ratio,ci_ratio,bi_ratio,ip_ratio,ic_ratio,avg_unique_ip_hour,max_unique_ip_hour,avg_unique_ip_day,max_unique_ip_day
113876,ACCT_198075,1442,0.331494,1442,15.356069,1,1521,0.349655,1521,1.0,0.000657,16,0.003678,"CRM,CDM",-4.918016e+06,-10703236.73,-4.956250e+06,-4.885021e+06,3763550.54,0.0,0.0,0.0,0.957241,0.037011,0.005747,0.0,1.0,3.0,2.0,7.0
89166,ACCT_050664,367,0.333333,367,15.529265,1,366,0.332425,366,1.0,0.002732,2,0.001817,"CRM,CDM",-1.164735e+07,-31345076.39,-1.178925e+07,-1.152843e+07,755366.47,0.0,0.0,0.0,0.940963,0.035422,0.023615,0.0,1.0,2.0,1.0,3.0
30461,ACCT_074490,2149,0.369181,2149,842.708552,1,1988,0.341522,1988,1.0,0.000503,15,0.002577,"CRM,CDM",2.294970e+07,-290285.62,2.289777e+07,2.299640e+07,37784979.45,0.0,0.0,0.0,0.956365,0.033328,0.010308,0.0,1.0,3.0,2.0,8.0
100552,ACCT_021242,1053,0.342997,1053,14.705851,1,1056,0.343974,1056,1.0,0.000947,8,0.002606,"CRM,CDM",-1.789307e+07,-35212283.86,-1.758815e+07,-1.815721e+07,141937.07,0.0,0.0,0.0,0.945277,0.035179,0.019544,0.0,1.0,2.0,1.0,4.0
55981,ACCT_114114,465,0.294490,465,14.923299,1,561,0.355288,561,1.0,0.001783,4,0.002533,"CRM,CDM",-2.092955e+06,-8756150.30,-2.145409e+06,-2.046098e+06,4327906.45,0.0,0.0,0.0,0.960735,0.033566,0.005700,0.0,1.0,3.0,2.0,8.0
57122,ACCT_142737,395,0.281540,395,15.336928,1,472,0.336422,472,1.0,0.002119,5,0.003564,"CRM,CDM",4.535080e+06,-205636.15,4.526043e+06,4.544002e+06,9474833.30,0.0,0.0,0.0,0.925160,0.032074,0.042766,0.0,1.0,2.0,1.0,3.0


In [32]:
import duckdb

con = duckdb.connect()

con.execute("""
COPY (
    SELECT *
    FROM 'transaction_features.parquet' t1
    LEFT JOIN 'transaction_features_2.parquet' t2 USING(account_id)
    LEFT JOIN 'transaction_features_3.parquet' t3 USING(account_id)
    LEFT JOIN 'transaction_features_4.parquet' t4 USING(account_id)
)
TO 'transaction_final.parquet'
(FORMAT PARQUET);
""")

In [33]:
df = pd.read_parquet("transaction_final.parquet")
print(df.shape)
df.sample(10)

(159776, 83)


,account_id,total_transactions,unique_mcc_count,most_common_mcc,max_mcc_frequency,mcc_max_amount,mcc_min_amount,unique_channel_count,most_common_channel,max_channel_frequency,channel_max_amount,channel_min_amount,avg_unique_mcc_per_hour,avg_unique_mcc_per_day,avg_unique_channel_per_hour,avg_unique_channel_per_day,least_common_channel,least_common_mcc_code,total_transaction_amount,avg_transaction_amount,max_transaction_amount,min_transaction_amount,std_transaction_amount,median_transaction_amount,total_credit_amount,total_debit_amount,max_credit_amount,max_debit_amount,credit_debit_ratio,credit_ratio,debit_ratio,avg_time_between_txn_sec,median_time_between_txn_sec,max_txn_1hr,max_txn_10min,max_txn_1day,max_txn_1week,avg_txn_per_day,avg_txn_per_hr,avg_txn_per_week,avg_txn_per_month,night_txn_rate,day_txn_rate,rapid_passthrough_count,unique_counterparty_count,repeated_counterparty_frequency,unique_credit_counterparty_count,unique_debit_counterparty_count,avg_unique_counterparty_1hr,max_unique_counterparty_1hr,avg_repeated_counterparty_freq_1hr,avg_unique_counterparty_1day,max_unique_counterparty_1day,avg_repeated_counterparty_freq_1day,geo_txn_count,geo_frac,unique_geo_count,geo_bbox_km,has_geo_flag,ip_txn_count,ip_frac,unique_ip_count,top_ip_count,top_ip_share,atm_deposit_count,atm_deposit_frac,atm_deposit_types,avg_balance,min_balance,avg_balance_credit,avg_balance_debit,max_balance,clt_cash_ratio,loan_ratio,normal_ratio,ci_ratio,bi_ratio,ip_ratio,ic_ratio,avg_unique_ip_hour,max_unique_ip_hour,avg_unique_ip_day,max_unique_ip_day
27394,ACCT_108653,2155,55,9384,459,5905,2557,34,UPC,836,UPD,UPD,1,2,1,2,IAD,4816,6.651004e+07,30863.126325,22743000.00,-17296.01,500688.895332,1000.0,38680467.26,2.782957e+07,22743000.00,2614245.02,1.389905,0.461717,0.538283,27210.097957,15473.0,3,2,11,34,3.173785,0.132363,22.216495,93.695652,0.051508,0.929930,9.0,15,0.993039,15,14,1.088325,3,0.002538,3.003125,8,0.079794,837,0.388399,837,15.138627,1,690,0.320186,690,1.0,0.001449,5,0.002320,CDM,1.226393e+07,-1758117.75,1.216182e+07,1.235151e+07,21338606.46,0.0,0.0,0.0,0.954060,0.035731,0.010209,0.000000,1.0,3.0,2.0,5.0
140190,ACCT_090871,1719,58,9384,330,2199,5905,32,UPC,610,IPM,UPD,1,2,1,2,SID,4816,5.752203e+07,33462.496451,7587560.64,-39795.56,278274.046602,1000.0,19954834.78,3.756720e+07,3720824.01,7587560.64,0.531177,0.457824,0.542176,29064.771828,16034.0,3,3,9,36,2.968912,0.123919,20.710843,85.950000,0.059919,0.926702,8.0,82,0.952298,48,60,1.096609,3,0.001493,3.001848,9,0.039618,569,0.331006,569,15.347558,1,596,0.346713,596,1.0,0.001678,4,0.002327,"CRM,CDM",-8.079206e+06,-17197966.17,-7.876391e+06,-8.250467e+06,3150187.13,0.0,0.0,0.0,0.948226,0.040721,0.011053,0.000000,1.0,2.0,2.0,5.0
24578,ACCT_051604,1700,56,9384,330,2557,2017,33,UPC,689,END,UPC,1,1,1,1,CTC,7995,3.213968e+07,18905.691276,3139400.00,-20913.83,115896.115317,1000.0,17427029.40,1.471265e+07,3139400.00,1213000.00,1.184493,0.478824,0.521176,84319.761036,64550.0,3,2,6,14,1.024714,0.042719,7.172996,30.909091,0.048235,0.937647,2.0,88,0.948235,28,86,1.027828,3,0.000302,1.594078,6,0.010220,530,0.311765,530,15.127721,1,556,0.327059,556,1.0,0.001799,2,0.001176,"CRM,CDM",2.945932e+06,-518502.78,3.121123e+06,2.784978e+06,5086292.62,0.0,0.0,0.0,0.940000,0.028235,0.031765,0.000000,1.0,2.0,1.0,3.0
139650,ACCT_041611,8464,58,9384,1804,2921,2943,35,UPC,3155,UPD,UPD,1,6,1,5,SCW,5251,1.955818e+08,23107.487192,11611600.00,-617192.96,187963.527445,1000.0,85491689.28,1.100901e+08,5433979.60,11611600.00,0.776561,0.460539,0.539461,6786.360983,3212.0,6,3,28,107,12.727820,0.530525,89.094737,384.727273,0.051630,0.928993,137.0,7,0.999173,7,6,1.356775,5,0.035858,5.148872,7,0.543978,2670,0.315454,2670,15.124945,1,2960,0.349716,2959,2.0,0.000676,28,0.003308,"CRM,CDM",-1.391158e+07,-31388258.02,-1.403041e+07,-1.381015e+07,300193.64,0.0,0.0,0.0,0.964083,0.033436,0.002481,0.000000,1.0,4.0,5.0,12.0
117237,ACCT_042054,1964,55,9384,400,5651,5651,33,UPC,712,UPC,MCR,1,1,1,2,RTD,6011,4.059163e+07,20667.834684,4405902.76,-1

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159776 entries, 0 to 159775
Data columns (total 83 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   account_id                           159776 non-null  object 
 1   total_transactions                   159776 non-null  int64  
 2   unique_mcc_count                     159776 non-null  int64  
 3   most_common_mcc                      159776 non-null  int64  
 4   max_mcc_frequency                    159776 non-null  int64  
 5   mcc_max_amount                       159776 non-null  int64  
 6   mcc_min_amount                       159776 non-null  int64  
 7   unique_channel_count                 159776 non-null  int64  
 8   most_common_channel                  159776 non-null  object 
 9   max_channel_frequency                159776 non-null  int64  
 10  channel_max_amount                   159776 non-null  object 
 11  channel_min_a

In [35]:
# check duplicate column names
df.columns[df.columns.duplicated()]

Index([], dtype='object')

In [39]:
import pandas as pd

pd.set_option('display.max_rows', None)

In [40]:
df.isnull().sum()

account_id                                 0
total_transactions                         0
unique_mcc_count                           0
most_common_mcc                            0
max_mcc_frequency                          0
mcc_max_amount                             0
mcc_min_amount                             0
unique_channel_count                       0
most_common_channel                        0
max_channel_frequency                      0
channel_max_amount                         0
channel_min_amount                         0
avg_unique_mcc_per_hour                    0
avg_unique_mcc_per_day                     0
avg_unique_channel_per_hour                0
avg_unique_channel_per_day                 0
least_common_channel                       0
least_common_mcc_code                      0
total_transaction_amount                   0
avg_transaction_amount                     0
max_transaction_amount                     0
min_transaction_amount                     0
std_transa